# CIFAKE Guard: Real vs AI-Generated Image Detection

This notebook trains and compares three architectures on the **CIFAKE** dataset
(60k real CIFAR-10 images + 60k Stable Diffusion generated images, 32x32 px).

| # | Model | Training strategy |
|---|-------|-------------------|
| 1 | **LightCNN** | trained from scratch |
| 2 | **EfficientNet-B0** (ImageNet weights) | fine-tuned |
| 3 | **TinyViT** | trained from scratch |

**Positive class = FAKE** (AI-generated), which is the class a detector is meant to catch.

**Runs anywhere:**
* On a CPU-only laptop, `QUICK_RUN` turns on automatically and uses a tiny stratified slice of CIFAKE.
* On Google Colab with a GPU runtime (*Runtime > Change runtime type > T4 GPU*), the full dataset is used.

Outputs:
* `plots/`: loss & accuracy curves (refreshed after **every epoch**), ROC / PR curves, confusion matrices
* `checkpoints/`: best weights for each model (by validation loss)
* `results/`: metrics table and per-epoch history as CSV

## 0. Setup

In [ ]:
# Install missing dependencies (Colab already has torch/torchvision/sklearn; kagglehub may need installing).
import importlib, subprocess, sys
for pkg in ["kagglehub", "sklearn", "pandas", "matplotlib", "tqdm", "truststore"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        name = "scikit-learn" if pkg == "sklearn" else pkg
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", name])

# Use the operating system's certificate store for HTTPS. This fixes
# "SSL: CERTIFICATE_VERIFY_FAILED" on laptops where antivirus/proxy software inspects HTTPS.
import truststore
truststore.inject_into_ssl()

In [ ]:
import os, time, random, math, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef, log_loss, brier_score_loss,
    confusion_matrix, roc_curve, precision_recall_curve, classification_report,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IN_COLAB = "google.colab" in sys.modules
print(f"PyTorch {torch.__version__} | device: {DEVICE} | Colab: {IN_COLAB}")

## 1. Configuration

In [ ]:
# QUICK_RUN = True  -> tiny subset, few epochs (CPU laptop)
# QUICK_RUN = False -> full CIFAKE (Colab GPU)
# Auto-detected from GPU availability; override manually if you like.
QUICK_RUN = not torch.cuda.is_available()

SEED = 42

if QUICK_RUN:
    TRAIN_PER_CLASS = 1000      # images per class used for training
    VAL_PER_CLASS   = 250       # held out from the official train split
    TEST_PER_CLASS  = 500       # sampled from the official test split
    EPOCHS          = 5
    BATCH_SIZE      = 64
else:
    TRAIN_PER_CLASS = None      # None = use everything (50k per class minus validation)
    VAL_PER_CLASS   = 5000
    TEST_PER_CLASS  = None      # full official test set (10k per class)
    EPOCHS          = 20
    BATCH_SIZE      = 256

EFFNET_IMG_SIZE = 96 if QUICK_RUN else 128   # CIFAKE is 32x32; EfficientNet needs upscaling
NUM_WORKERS     = 0 if os.name == "nt" else min(4, os.cpu_count() or 1)   # Windows notebooks cannot fork workers
PATIENCE        = 5          # early stopping on validation loss

PLOTS_DIR   = Path("plots");       PLOTS_DIR.mkdir(exist_ok=True)
CKPT_DIR    = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path("results");     RESULTS_DIR.mkdir(exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()
print(f"QUICK_RUN={QUICK_RUN} | epochs={EPOCHS} | batch={BATCH_SIZE} | workers={NUM_WORKERS}")

## 2. Download the CIFAKE dataset

Source: [Kaggle: birdy654/cifake-real-and-ai-generated-synthetic-images](https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images) (~105 MB).
`kagglehub` downloads public datasets without credentials and caches them, so re-running is instant.

Expected layout: `train/{REAL,FAKE}` (50k each) and `test/{REAL,FAKE}` (10k each).

In [ ]:
import kagglehub

dataset_path = Path(kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images"))

def find_split_root(base: Path, split: str) -> Path:
    # Locate the folder that contains REAL/ and FAKE/ for a split (robust to nesting).
    for p in [base / split, *base.rglob(split)]:
        if p.is_dir() and (p / "REAL").is_dir() and (p / "FAKE").is_dir():
            return p
    raise FileNotFoundError(f"Could not find '{split}/REAL' and '{split}/FAKE' under {base}")

TRAIN_DIR = find_split_root(dataset_path, "train")
TEST_DIR  = find_split_root(dataset_path, "test")

for d in (TRAIN_DIR, TEST_DIR):
    counts = {c: len(os.listdir(d / c)) for c in ("REAL", "FAKE")}
    print(f"{d}: {counts}")

## 3. Data pipeline

* A stratified subset is sampled **once** (fixed seed), so every model sees exactly the same images.
* The validation set is carved out of the official train split. The official test split is never used for model selection.
* Augmentation is kept mild (flip + small translation). Heavy colour/blur augmentation would destroy the subtle generation artefacts the detector relies on.

In [ ]:
_probe = datasets.ImageFolder(TRAIN_DIR)
CLASS_TO_IDX = _probe.class_to_idx            # {'FAKE': 0, 'REAL': 1} (alphabetical)
FAKE_IDX = CLASS_TO_IDX["FAKE"]
print("ImageFolder classes:", CLASS_TO_IDX, "-> positive class (label 1) = FAKE")

def stratified_indices(targets, per_class, rng, exclude=None):
    # Pick `per_class` indices for each class (None = all remaining).
    targets = np.asarray(targets)
    keep = np.ones(len(targets), dtype=bool)
    if exclude is not None:
        keep[np.asarray(exclude, dtype=int)] = False
    chosen = []
    for c in np.unique(targets):
        idx = np.where((targets == c) & keep)[0]
        rng.shuffle(idx)
        chosen.extend(idx if per_class is None else idx[:per_class])
    chosen = np.array(chosen)
    rng.shuffle(chosen)
    return chosen.tolist()

rng = np.random.default_rng(SEED)
train_targets = _probe.targets
test_targets  = datasets.ImageFolder(TEST_DIR).targets

VAL_IDX   = stratified_indices(train_targets, VAL_PER_CLASS, rng)
TRAIN_IDX = stratified_indices(train_targets, TRAIN_PER_CLASS, rng, exclude=VAL_IDX)
TEST_IDX  = stratified_indices(test_targets,  TEST_PER_CLASS, rng)
assert not set(VAL_IDX) & set(TRAIN_IDX), "train/val overlap!"
print(f"train={len(TRAIN_IDX)} | val={len(VAL_IDX)} | test={len(TEST_IDX)}")

CIFAKE_MEAN, CIFAKE_STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def build_transforms(img_size=32, mean=CIFAKE_MEAN, std=CIFAKE_STD):
    resize = [transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.BICUBIC)] if img_size != 32 else []
    train_tf = transforms.Compose(resize + [
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    eval_tf = transforms.Compose(resize + [transforms.ToTensor(), transforms.Normalize(mean, std)])
    return train_tf, eval_tf

def make_loaders(img_size=32, mean=CIFAKE_MEAN, std=CIFAKE_STD, batch_size=BATCH_SIZE):
    train_tf, eval_tf = build_transforms(img_size, mean, std)
    train_ds = Subset(datasets.ImageFolder(TRAIN_DIR, transform=train_tf), TRAIN_IDX)
    val_ds   = Subset(datasets.ImageFolder(TRAIN_DIR, transform=eval_tf),  VAL_IDX)
    test_ds  = Subset(datasets.ImageFolder(TEST_DIR,  transform=eval_tf),  TEST_IDX)
    kw = dict(batch_size=batch_size, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
              persistent_workers=NUM_WORKERS > 0)
    return (DataLoader(train_ds, shuffle=True, drop_last=True, **kw),
            DataLoader(val_ds, shuffle=False, **kw),
            DataLoader(test_ds, shuffle=False, **kw))

def to_binary(y):
    # Map ImageFolder labels to 1 = FAKE, 0 = REAL.
    return (y == FAKE_IDX).float()

In [ ]:
# Quick visual sanity check
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for row, cls in enumerate(["REAL", "FAKE"]):
    cls_idx = [i for i in TRAIN_IDX if _probe.targets[i] == CLASS_TO_IDX[cls]][:8]
    for ax, i in zip(axes[row], cls_idx):
        ax.imshow(_probe[i][0]); ax.set_title(cls, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig(PLOTS_DIR / "sample_images.png", dpi=120); plt.show()

## 4. Model 1: LightCNN (from scratch)

Four conv stages (32 > 64 > 128 > 256 channels) with BatchNorm, ReLU and max-pooling, followed by global average pooling.
Global pooling keeps the parameter count low and avoids a large fully-connected layer. The model has roughly 0.6 M parameters.

In [ ]:
def conv_bn(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

class LightCNN(nn.Module):
    def __init__(self, num_outputs=1, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            conv_bn(3, 32),   conv_bn(32, 32),   nn.MaxPool2d(2),   # 32 -> 16
            conv_bn(32, 64),  conv_bn(64, 64),   nn.MaxPool2d(2),   # 16 -> 8
            conv_bn(64, 128), conv_bn(128, 128), nn.MaxPool2d(2),   # 8 -> 4
            conv_bn(128, 256),                                       # 4x4
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(256, num_outputs))

    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)   # raw logit (BCEWithLogits)

def count_params(m, trainable_only=False):
    return sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable_only)

print(f"LightCNN params: {count_params(LightCNN())/1e6:.3f} M")

## 5. Model 2: EfficientNet-B0 (fine-tuned)

B0 is the smallest EfficientNet (about 4 M parameters after replacing the head) and suits a 2-class task with low-resolution inputs.
* CIFAKE images are upscaled from 32 px to `EFFNET_IMG_SIZE`. At 32 px the 32x downsampling would collapse the feature map to 1x1.
* **Two-phase fine-tuning:** the backbone is frozen for the first `FREEZE_EPOCHS` epochs so the new head can warm up. Then everything is unfrozen, and the backbone gets a 10x lower learning rate than the head (discriminative learning rates).

In [ ]:
class EfficientNetB0Detector(nn.Module):
    def __init__(self, num_outputs=1, pretrained=True, dropout=0.3):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        net = models.efficientnet_b0(weights=weights)
        self.features = net.features
        self.pool = net.avgpool
        in_feats = net.classifier[1].in_features   # 1280
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(in_feats, num_outputs))

    def set_backbone_trainable(self, trainable: bool):
        for p in self.features.parameters():
            p.requires_grad = trainable

    def param_groups(self, lr):
        return [{"params": self.features.parameters(), "lr": lr * 0.1},
                {"params": self.head.parameters(),     "lr": lr}]

    def forward(self, x):
        return self.head(self.pool(self.features(x))).squeeze(1)

FREEZE_EPOCHS = 1
print(f"EfficientNet-B0 params: {count_params(EfficientNetB0Detector(pretrained=False))/1e6:.3f} M")

## 6. Model 3: TinyViT (from scratch)

A minimal Vision Transformer sized for 32x32 inputs:
* 4x4 patches give 64 tokens, plus one [CLS] token and learned positional embeddings.
* 6 pre-norm transformer blocks, embedding dimension 192, 3 heads, MLP ratio 2. That is roughly 1.8 M parameters.
* ViTs lack the inductive biases of CNNs, so on small data they need warmup, weight decay and label smoothing to train well. All three are applied in the training call.

In [ ]:
class TinyViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, embed_dim=192, depth=6, num_heads=3,
                 mlp_ratio=2.0, dropout=0.1, num_outputs=1):
        super().__init__()
        assert img_size % patch_size == 0
        n_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_outputs)
        self.apply(self._init_weights)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)          # (B, N, D)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = self.pos_drop(torch.cat([cls, x], dim=1) + self.pos_embed)
        x = self.norm(self.encoder(x))
        return self.head(x[:, 0]).squeeze(1)                         # [CLS] token logit

print(f"TinyViT params: {count_params(TinyViT())/1e6:.3f} M")

## 7. Training loop

Shared by all three models:
* `BCEWithLogitsLoss` on a single logit, with optional label smoothing
* AdamW and a linear-warmup + cosine-decay schedule, stepped per batch
* Mixed precision (AMP) on GPU, gradient clipping always
* Early stopping on validation loss, with the best checkpoint saved to `checkpoints/`
* **Loss and accuracy curves are re-drawn and saved to `plots/` after every epoch** (`<model>_loss.png`, `<model>_accuracy.png`)

In [ ]:
def save_epoch_plots(history, model_name):
    # Overwrite plots/<model>_loss.png and plots/<model>_accuracy.png with the curves so far.
    epochs = range(1, len(history["train_loss"]) + 1)
    for metric, ylabel, fname in [("loss", "BCE loss", "loss"), ("acc", "Accuracy", "accuracy")]:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(epochs, history[f"train_{metric}"], marker="o", label="train")
        ax.plot(epochs, history[f"val_{metric}"],   marker="o", label="validation")
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_title(f"{model_name}: {ylabel} (epoch {len(history['train_loss'])})")
        ax.grid(alpha=0.3); ax.legend()
        fig.tight_layout(); fig.savefig(PLOTS_DIR / f"{model_name}_{fname}.png", dpi=120)
        plt.close(fig)

def warmup_cosine(optimizer, total_steps, warmup_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def run_epoch(model, loader, criterion, optimizer=None, scheduler=None, scaler=None,
              label_smoothing=0.0, desc=""):
    train = optimizer is not None
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    use_amp = DEVICE.type == "cuda"
    pbar = tqdm(loader, desc=desc, leave=False)
    for x, y in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = to_binary(y).to(DEVICE, non_blocking=True)
        y_target = y * (1 - label_smoothing) + 0.5 * label_smoothing if train else y
        with torch.set_grad_enabled(train), torch.autocast(device_type=DEVICE.type, enabled=use_amp):
            logits = model(x)
            loss = criterion(logits.float(), y_target)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            if scheduler is not None: scheduler.step()
        bs = y.size(0)
        total_loss += loss.item() * bs
        correct += ((logits.detach() > 0).float() == y).sum().item()
        n += bs
        pbar.set_postfix(loss=f"{total_loss/n:.4f}", acc=f"{correct/n:.4f}")
    return total_loss / n, correct / n

def train_model(model, model_name, loaders, epochs=EPOCHS, lr=1e-3, weight_decay=0.05,
                warmup_epochs=1, label_smoothing=0.0, freeze_epochs=0, patience=PATIENCE):
    train_loader, val_loader, _ = loaders
    model = model.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    params = model.param_groups(lr) if hasattr(model, "param_groups") else model.parameters()
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    steps_per_epoch = len(train_loader)
    scheduler = warmup_cosine(optimizer, epochs * steps_per_epoch, warmup_epochs * steps_per_epoch)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=DEVICE.type == "cuda")

    history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "epoch_time_s"]}
    best_val, best_state, bad_epochs = float("inf"), None, 0
    ckpt_path = CKPT_DIR / f"{model_name}_best.pt"
    print(f"\n=== Training {model_name} | {count_params(model)/1e6:.3f} M params | {epochs} epochs ===")

    for epoch in range(1, epochs + 1):
        if freeze_epochs and hasattr(model, "set_backbone_trainable"):
            model.set_backbone_trainable(epoch > freeze_epochs)
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, scheduler, scaler,
                                    label_smoothing, desc=f"{model_name} ep{epoch} train")
        va_loss, va_acc = run_epoch(model, val_loader, criterion, desc=f"{model_name} ep{epoch} val")
        dt = time.time() - t0

        for k, v in zip(history, [tr_loss, tr_acc, va_loss, va_acc, optimizer.param_groups[-1]["lr"], dt]):
            history[k].append(v)
        save_epoch_plots(history, model_name)

        improved = va_loss < best_val
        if improved:
            best_val, bad_epochs = va_loss, 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, ckpt_path)
        else:
            bad_epochs += 1
        print(f"[{model_name}] epoch {epoch:02d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {va_loss:.4f} acc {va_acc:.4f} | {dt:.1f}s {'*best*' if improved else ''}")
        if bad_epochs >= patience:
            print(f"Early stopping: no val-loss improvement for {patience} epochs.")
            break

    model.load_state_dict(best_state)   # restore best weights for evaluation
    pd.DataFrame(history).rename_axis("epoch").to_csv(RESULTS_DIR / f"{model_name}_history.csv")
    return model, history

## 8. Train the three models

In [ ]:
# 8.1 LightCNN: native 32x32
cnn_loaders = make_loaders(img_size=32)
cnn_model, cnn_hist = train_model(
    LightCNN(), "LightCNN", cnn_loaders, lr=2e-3, weight_decay=5e-4, warmup_epochs=1)

In [ ]:
# 8.2 EfficientNet-B0: upscaled inputs, ImageNet normalisation, frozen backbone for the first epoch
eff_loaders = make_loaders(img_size=EFFNET_IMG_SIZE, mean=IMAGENET_MEAN, std=IMAGENET_STD,
                           batch_size=32 if QUICK_RUN else BATCH_SIZE)
eff_model, eff_hist = train_model(
    EfficientNetB0Detector(pretrained=True), "EfficientNetB0", eff_loaders,
    lr=1e-3, weight_decay=1e-4, warmup_epochs=1, freeze_epochs=FREEZE_EPOCHS)

In [ ]:
# 8.3 TinyViT: native 32x32, warmup + label smoothing + strong weight decay
vit_loaders = make_loaders(img_size=32)
vit_model, vit_hist = train_model(
    TinyViT(), "TinyViT", vit_loaders, lr=1e-3, weight_decay=0.05,
    warmup_epochs=1 if QUICK_RUN else 3, label_smoothing=0.1)

## 9. Validation & evaluation (held-out test set)

Metrics reported for each model, with FAKE as the positive class:

| Metric | Why it matters for a fake-image detector |
|---|---|
| Accuracy / Balanced accuracy | Overall correctness. The balanced variant is robust to class imbalance. |
| Precision | Of the images flagged as fake, the fraction that really are fake (false-alarm cost). |
| Recall / TPR | Fraction of fakes caught (miss cost). |
| Specificity / TNR | Fraction of real images correctly passed. |
| F1 | Harmonic mean of precision and recall. |
| MCC | Single balanced score that uses the whole confusion matrix. |
| ROC-AUC | Threshold-independent ranking quality. |
| PR-AUC (Average Precision) | Ranking quality focused on the positive class. |
| **EER** | Equal Error Rate, where FPR = FNR. Standard in biometrics and forensics. |
| **TPR @ 1% FPR** | Detection rate when only 1 in 100 real images may be falsely flagged (an operational setting). |
| Log loss / Brier score | Calibration quality of the predicted probabilities. |
| Latency / throughput / size | Deployment cost. |

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval().to(DEVICE)
    probs, labels = [], []
    for x, y in tqdm(loader, desc="predict", leave=False):
        logits = model(x.to(DEVICE))
        probs.append(torch.sigmoid(logits.float()).cpu())
        labels.append(to_binary(y))
    return torch.cat(labels).numpy().astype(int), torch.cat(probs).numpy()

@torch.no_grad()
def measure_latency(model, img_size, n_warmup=5, n_runs=30, batch=64):
    model.eval().to(DEVICE)
    def timed(bs, runs):
        x = torch.randn(bs, 3, img_size, img_size, device=DEVICE)
        for _ in range(n_warmup): model(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs): model(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        return (time.perf_counter() - t0) / runs
    single_ms = timed(1, n_runs) * 1000
    throughput = batch / timed(batch, max(5, n_runs // 3))
    return single_ms, throughput

def model_size_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2

def evaluate(model, name, loader, img_size, threshold=0.5):
    y_true, y_prob = predict(model, loader)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fnr = 1 - tpr
    eer_i = np.nanargmin(np.abs(fnr - fpr))
    eer = (fpr[eer_i] + fnr[eer_i]) / 2
    tpr_at_1fpr = float(np.interp(0.01, fpr, tpr))
    latency_ms, throughput = measure_latency(model, img_size)
    metrics = {
        "model": name,
        "params_M": count_params(model) / 1e6,
        "size_MB": model_size_mb(model),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_TPR": recall_score(y_true, y_pred, zero_division=0),
        "specificity_TNR": tn / (tn + fp) if (tn + fp) else 0.0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "eer": eer,
        "tpr@1%fpr": tpr_at_1fpr,
        "log_loss": log_loss(y_true, np.clip(y_prob, 1e-7, 1 - 1e-7), labels=[0, 1]),
        "brier": brier_score_loss(y_true, y_prob),
        "latency_ms_bs1": latency_ms,
        "throughput_img_s": throughput,
    }
    print(f"\n----- {name} -----")
    print(classification_report(y_true, y_pred, target_names=["REAL", "FAKE"], digits=4, zero_division=0))
    return metrics, (y_true, y_prob, y_pred)

In [ ]:
eval_specs = [
    ("LightCNN",       cnn_model, cnn_loaders[2], 32),
    ("EfficientNetB0", eff_model, eff_loaders[2], EFFNET_IMG_SIZE),
    ("TinyViT",        vit_model, vit_loaders[2], 32),
]
all_metrics, all_preds = [], {}
for name, model, test_loader, img_size in eval_specs:
    m, preds = evaluate(model, name, test_loader, img_size)
    all_metrics.append(m); all_preds[name] = preds

results_df = pd.DataFrame(all_metrics).set_index("model")
results_df.to_csv(RESULTS_DIR / "test_metrics.csv")
pd.set_option("display.float_format", "{:.4f}".format)
results_df

## 10. Comparison plots (ROC, Precision-Recall, confusion matrices, training curves)

In [ ]:
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5))
for name, (y_true, y_prob, _) in all_preds.items():
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ax_roc.plot(fpr, tpr, label=f"{name} (AUC={results_df.loc[name, 'roc_auc']:.4f})")
    ax_pr.plot(rec, prec, label=f"{name} (AP={results_df.loc[name, 'pr_auc']:.4f})")
ax_roc.plot([0, 1], [0, 1], "k--", lw=0.8)
ax_roc.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC curve (test)")
ax_pr.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall curve (test)")
ax_roc.legend(loc="lower right"); ax_pr.legend(loc="lower left")
for ax in (ax_roc, ax_pr): ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(PLOTS_DIR / "roc_pr_curves.png", dpi=120); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, (y_true, _, y_pred)) in zip(axes, all_preds.items()):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}\n({cm_norm[i, j]:.1%})", ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black")
    ax.set_xticks([0, 1], ["REAL", "FAKE"]); ax.set_yticks([0, 1], ["REAL", "FAKE"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(name)
plt.tight_layout(); plt.savefig(PLOTS_DIR / "confusion_matrices.png", dpi=120); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, hist in [("LightCNN", cnn_hist), ("EfficientNetB0", eff_hist), ("TinyViT", vit_hist)]:
    ep = range(1, len(hist["val_loss"]) + 1)
    axes[0].plot(ep, hist["val_loss"], marker="o", label=name)
    axes[1].plot(ep, hist["val_acc"],  marker="o", label=name)
axes[0].set(xlabel="Epoch", ylabel="Validation loss", title="Validation loss: all models")
axes[1].set(xlabel="Epoch", ylabel="Validation accuracy", title="Validation accuracy: all models")
for ax in axes: ax.grid(alpha=0.3); ax.legend(); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout(); plt.savefig(PLOTS_DIR / "all_models_validation_curves.png", dpi=120); plt.show()

print("Saved plots:", sorted(p.name for p in PLOTS_DIR.glob("*.png")))

## 11. Saving outputs when running on Colab

Colab's disk is wiped when the runtime ends. Run the cell below to zip `plots/`, `checkpoints/` and `results/` and download them.
You could also mount Google Drive and copy the folders there.

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    for folder in ["plots", "checkpoints", "results"]:
        shutil.make_archive(folder, "zip", folder)
        files.download(f"{folder}.zip")
else:
    print("Not on Colab: outputs are already in ./plots, ./checkpoints and ./results")